# brypt0-vol-radar

A small experiment with TimesFM-3. The end goal is to forecast volatility and
volume for BTC, ETH, and SOL, but I am building and checking one piece at a time.

The notebook currently covers a synthetic model check and the first real-data download.

## Before running

In Colab, go to **Runtime > Change runtime type** and choose a **T4 GPU**.
Then use **Runtime > Run all**.

The install and first model download can take a few minutes. The weights are
downloaded to Colab's temporary cache, not this GitHub repo. No API key is
needed for the public checkpoint.

In [ ]:
%pip install -q "timesfm[torch]==3.0.1" "pandas>=2.2,<3" "requests>=2.32,<3"

## Quick runtime check

A T4 is the intended Colab setup. The small smoke test can fall back to CPU,
which is handy when checking the notebook locally.

In [ ]:
import numpy as np
import torch
from timesfm3 import TimesFM3Forecaster

np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU fallback is fine for this small test. Use a T4 in Colab.")

## Make some fake hourly data

There are six targets: a volatility-like and volume-like series for each
asset. The values are made up, but they have daily seasonality and a fake
event bump. This lets us check the multivariate and covariate plumbing before
involving real market data.

In [ ]:
ASSETS = ("BTC", "ETH", "SOL")
TARGETS = ("realized_volatility", "volume")
MODEL_ID = "google/timesfm-3.0-pytorch"
CONTEXT_HOURS = 128
HORIZON_HOURS = 24

total_hours = CONTEXT_HOURS + HORIZON_HOURS
hours = np.arange(total_hours)
hour_of_day = hours % 24

daily_sin = np.sin(2 * np.pi * hour_of_day / 24).astype(np.float32)
daily_cos = np.cos(2 * np.pi * hour_of_day / 24).astype(np.float32)

# A pretend scheduled announcement, known both before and after the cutoff.
event_flag = np.zeros(total_hours, dtype=np.float32)
event_flag[72:76] = 1
event_flag[136:140] = 1

rng = np.random.default_rng(42)
target_rows = []
target_names = []

for asset_number, asset in enumerate(ASSETS, start=1):
    volatility = (
        0.015 * asset_number
        + 0.003 * daily_sin
        + 0.012 * event_flag
        + rng.normal(0, 0.001, total_hours)
    )
    volume = (1_000_000_000 / asset_number) * (
        1
        + 0.20 * daily_cos
        + 0.45 * event_flag
        + rng.normal(0, 0.04, total_hours)
    )

    target_rows.extend([
        np.clip(volatility[:CONTEXT_HOURS], 0, None),
        np.clip(volume[:CONTEXT_HOURS], 0, None),
    ])
    target_names.extend([f"{asset}_volatility", f"{asset}_volume"])

context = np.asarray(target_rows, dtype=np.float32)
known_future = np.asarray(
    [daily_sin, daily_cos, event_flag],
    dtype=np.float32,
)

print("Targets:", context.shape)
print("Known-future covariates:", known_future.shape)
print("Series:", ", ".join(target_names))

## Load TimesFM-3 and forecast 24 hours

One 2D context array means the six series are forecast together. The three
covariates include their known values through the forecast horizon.

In [ ]:
forecaster = TimesFM3Forecaster.from_pretrained(
    MODEL_ID,
    device=device,
    per_core_batch_size=1,
)

prediction = forecaster.predict(
    context=context,
    horizon=HORIZON_HOURS,
    past_future_covariates=known_future,
    return_quantiles=True,
    make_positive=True,
)

print("Point forecast:", prediction.forecast.shape)
print("Quantiles:", prediction.quantiles.shape)

## Check the result

TimesFM-3 returns the median forecast plus nine quantiles (p10 through p90).
These checks are intentionally boring: right shape, real numbers, ordered
intervals, and the point forecast matching p50.

In [ ]:
expected_forecast_shape = (len(target_names), HORIZON_HOURS)
expected_quantile_shape = (len(target_names), HORIZON_HOURS, 9)

assert prediction.forecast.shape == expected_forecast_shape
assert prediction.quantiles.shape == expected_quantile_shape
assert np.isfinite(prediction.forecast).all()
assert np.isfinite(prediction.quantiles).all()
assert (np.diff(prediction.quantiles, axis=-1) >= 0).all()
np.testing.assert_allclose(
    prediction.forecast,
    prediction.quantiles[..., 4],
    rtol=1e-5,
    atol=1e-6,
)

print(f"Synthetic forecast check passed on {device}.")
print("Forecast shape:", prediction.forecast.shape)
print("Quantile shape:", prediction.quantiles.shape)

## Synthetic test result

The six-series synthetic forecast passed with the real model. Synthetic accuracy
does not tell us whether the crypto forecasts will be useful, but the API and
array shapes work.

The TimesFM-3 weights currently use Google's non-commercial license, so this is
a learning project rather than a production trading system.


## Download hourly candles

I am using Coinbase Exchange because its candle endpoint is public and does
not need an API key. All three products are quoted in USD.

Coinbase limits custom requests to 300 candles, so this fetches 180 days in
299-hour chunks. Missing buckets stay missing for now. I will deal with them when calculating
returns and rolling features.

This short version is kept in the notebook so it works by itself in Colab.
The reusable command-line version is in `crypto_radar/data.py`.

In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests

COINBASE_API = "https://api.exchange.coinbase.com"
PRODUCTS = {"BTC": "BTC-USD", "ETH": "ETH-USD", "SOL": "SOL-USD"}
COLUMNS = ["timestamp", "low", "high", "open", "close", "volume"]


def fetch_coinbase_hours(product_id, start, end, pause=0.15):
    rows = []
    cursor = start
    headers = {"User-Agent": "brypt0-vol-radar/0.1"}

    while cursor < end:
        chunk_end = min(cursor + pd.Timedelta(hours=299), end)
        params = {
            "start": cursor.isoformat(),
            "end": chunk_end.isoformat(),
            "granularity": 3600,
        }

        for attempt in range(3):
            response = requests.get(
                f"{COINBASE_API}/products/{product_id}/candles",
                params=params,
                headers=headers,
                timeout=30,
            )
            if response.status_code != 429 and response.status_code < 500:
                break
            time.sleep(2**attempt)

        response.raise_for_status()
        rows.extend(response.json())
        cursor = chunk_end
        time.sleep(pause)

    candles = pd.DataFrame(rows, columns=COLUMNS)
    candles["timestamp"] = pd.to_datetime(candles["timestamp"], unit="s", utc=True)
    candles = candles[(candles["timestamp"] >= start) & (candles["timestamp"] < end)]
    candles = candles.drop_duplicates("timestamp").sort_values("timestamp")
    for column in COLUMNS[1:]:
        candles[column] = pd.to_numeric(candles[column], errors="raise")
    return candles.reset_index(drop=True)

In [ ]:
end_hour = pd.Timestamp.now(tz="UTC").floor("h")
start_hour = end_hour - pd.Timedelta(days=180)
frames = []

for asset, product_id in PRODUCTS.items():
    print(f"Downloading {product_id}...")
    candles = fetch_coinbase_hours(product_id, start_hour, end_hour)
    candles.insert(0, "product_id", product_id)
    candles.insert(0, "asset", asset)
    frames.append(candles)

market_data = pd.concat(frames, ignore_index=True)
market_data = market_data.sort_values(["asset", "timestamp"]).reset_index(drop=True)

output_path = Path("data/hourly_market_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
market_data.to_csv(output_path, index=False)
print(f"Saved {len(market_data):,} rows to {output_path}")


## Check the download

The API can have missing intervals. I want to see those gaps instead of
silently forward-filling them. Volume is in the base asset here, so BTC,
ETH, and SOL volume values are not directly comparable yet.

In [ ]:
assert set(market_data["asset"]) == set(PRODUCTS)
assert not market_data.duplicated(["asset", "timestamp"]).any()
assert market_data[COLUMNS[1:]].notna().all().all()
assert (market_data[["open", "high", "low", "close"]] > 0).all().all()
assert (market_data["volume"] >= 0).all()
assert (market_data["high"] >= market_data[["open", "close", "low"]].max(axis=1)).all()
assert (market_data["low"] <= market_data[["open", "close", "high"]].min(axis=1)).all()

expected_hours = pd.date_range(start_hour, end_hour, freq="h", inclusive="left")
summary_rows = []
missing_by_asset = {}

for asset, group in market_data.groupby("asset", sort=True):
    timestamps = pd.DatetimeIndex(group["timestamp"])
    missing = expected_hours.difference(timestamps)
    missing_by_asset[asset] = set(missing)
    summary_rows.append({
        "asset": asset,
        "rows": len(group),
        "first_hour": timestamps.min(),
        "last_hour": timestamps.max(),
        "missing_hours": len(missing),
        "duplicate_hours": int(timestamps.duplicated().sum()),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

shared_missing = set.intersection(*missing_by_asset.values())
print(f"Data check passed: {len(market_data):,} hourly candles downloaded.")
print(f"Hours missing for all three assets: {len(shared_missing)}")
print("The CSV is ready for feature work.")


## Feature work

Next I will inspect the missing hours, calculate log returns, build rolling
24-hour realized volatility, and put volume on a more useful scale.

Data source: [Coinbase Exchange candles](https://docs.cdp.coinbase.com/api-reference/exchange-api/rest-api/products/get-product-candles)
(checked September 4, 2026).